# 0️⃣ Introduction
We are going to work on a dataset from the Instacart Market Basket Analysis Kaggle competition. See the [data](https://www.kaggle.com/competitions/instacart-market-basket-analysis/data) page on Kaggle for more information on the different tables. The tables are relatively small, so an in-memory transformations library like Pandas might make more sense. This combined with the fact that the community version of Databricks is quite limited in terms of compute might make the transformations a bit slower then you would expect. Nevertheless, we are here to get familiar with the syntax of Pyspark, so lets get started! 🚀

In [0]:
# import the libraries
import pyspark
from pyspark.sql import functions as F

# 1️⃣ Loading the data
⏱ This should take about 1.23 minutes. The data is loaded from Amazon S3.

In [0]:
orders_large = spark.read.csv('s3://wagon-public-datasets/data-engineering/W3D3-processing/data/orders.csv', header='true',
                      inferSchema='true')
orderContents_large = spark.read.csv('s3://wagon-public-datasets/data-engineering/W3D3-processing/data/order_products__prior.csv', header='true',
                      inferSchema='true')
products_large = spark.read.csv('s3://wagon-public-datasets/data-engineering/W3D3-processing/data/products.csv', header='true',
                      inferSchema='true')

😅 This is a bit much data to develop with so lets create dataframes with 1/500th of the data to develop with and then you can apply functions to the larger data when you are ready!

In [0]:
split_weights = [1.0] * 500
orders = orders_large.randomSplit(split_weights)[0]
orderContents = orderContents_large.randomSplit(split_weights)[0]
products = products_large.randomSplit(split_weights)[0]

# 2️⃣ Prepping data
Let's get a little bit familiar with the PySpark syntax. 📃 Use the documentation of PySpark to help you.

❓**Aisles**

1️⃣ Load in the aisles table

2️⃣ Show the first 5 rows 

3️⃣ Print the schema

4️⃣ Cast data to correct types

5️⃣ Save as a temporary view

6️⃣ Select the data from the temporary view using SQL

In [0]:
aisles = spark.read.csv('s3://wagon-public-datasets/data-engineering/W3D3-processing/data/aisles.csv', header='true')
pass  # YOUR CODE HERE
# 2. Mostrar las primeras 5 filas
aisles.show(5)

# 3. Imprimir el esquema
aisles.printSchema()

# 4. Castear datos a tipos correctos (aisle_id suele ser entero)
aisles = aisles.withColumn("aisle_id", F.col("aisle_id").cast("integer"))

# 5. Guardar como una vista temporal
aisles.createOrReplaceTempView("aisles_view")

# 6. Seleccionar los datos de la vista usando SQL
spark.sql("SELECT * FROM aisles_view").show(5)

+--------+--------------------+
|aisle_id|               aisle|
+--------+--------------------+
|       1|prepared soups sa...|
|       2|   specialty cheeses|
|       3| energy granola bars|
|       4|       instant foods|
|       5|marinades meat pr...|
+--------+--------------------+
only showing top 5 rows
root
 |-- aisle_id: string (nullable = true)
 |-- aisle: string (nullable = true)

+--------+--------------------+
|aisle_id|               aisle|
+--------+--------------------+
|       1|prepared soups sa...|
|       2|   specialty cheeses|
|       3| energy granola bars|
|       4|       instant foods|
|       5|marinades meat pr...|
+--------+--------------------+
only showing top 5 rows


# 2️⃣ Analysis

### Get the historical orders of each user 
❓ Filter the orders table on the data where `eval_set` is equal `prior`. These are the historical baskets for all users.

In [0]:
pass  # YOUR CODE HERE
# Filtrar la tabla de órdenes donde eval_set sea 'prior'
orders_prior = orders.filter(F.col("eval_set") == "prior")

### Average basket size
❓ Calculate the mean size of the historical baskets that you filtered out here 👆 for all users. Join the historical baskets with the orderContents table and calculate the average basket size.

In [0]:
pass  # YOUR CODE HERE
# 1. Contar cuántos productos hay por cada order_id en orderContents
basket_sizes = orderContents.groupBy("order_id").count()

# 2. Unir con las órdenes históricas y calcular el promedio por usuario
avg_basket_size = orders_prior.join(basket_sizes, "order_id") \
    .groupBy("user_id") \
    .agg(F.mean("count").alias("average_basket_size"))

avg_basket_size.show(5)

+-------+-------------------+
|user_id|average_basket_size|
+-------+-------------------+
| 137813|                1.0|
| 102488|                1.0|
| 179170|                1.0|
|  77456|                1.0|
|  35422|                1.0|
+-------+-------------------+
only showing top 5 rows


### Total number of distinct items bought
❓Calculate the number of distinct items bought by each user

In [0]:
pass  # YOUR CODE HERE
# Unir órdenes históricas con contenidos y contar productos distintos por usuario
distinct_items = orders_prior.join(orderContents, "order_id") \
    .groupBy("user_id") \
    .agg(F.countDistinct("product_id").alias("total_distinct_items"))

distinct_items.show(5)

+-------+--------------------+
|user_id|total_distinct_items|
+-------+--------------------+
| 148346|                   1|
|  88741|                   1|
|  75375|                   1|
|  25077|                   1|
| 185577|                   2|
+-------+--------------------+
only showing top 5 rows


### What was the most recent order of each user and which products did this order contain? 

❓ 1️⃣ Get the last basket of each user from the orders table

❓ 2️⃣ Get the products from each of these orders from the orderContents table

In [0]:
pass  # YOUR CODE HERE
# 1. Obtener la última orden (máximo order_number) por usuario
last_order_id = orders.groupBy("user_id").agg(F.max("order_number").alias("order_number"))

# 2. Unir para obtener el order_id real de esa última orden
last_orders_info = last_order_id.join(orders, ["user_id", "order_number"])

# 3. Obtener los productos de esas órdenes finales
last_order_products = last_orders_info.join(orderContents, "order_id") \
    .select("user_id", "order_id", "product_id")

last_order_products.show(5)

+-------+--------+----------+
|user_id|order_id|product_id|
+-------+--------+----------+
|   5692| 3091391|      8518|
|  22584|  210597|      1930|
|   2978| 2092756|     43789|
|  10075| 2548076|     35727|
|   2601| 1195513|      9065|
+-------+--------+----------+
only showing top 5 rows


### Can you find the gluten-free products and return those?
Products that contain something like gluten/wheat free are considered gluten-free. Knowing which users have previously ordered gluten-free products could be useful for product recommendations in future machine learning models.

❓ 1️⃣ Identify the gluten-free products by looking at the `product_name` column in the `products` table

In [0]:
pass  # YOUR CODE HERE
# Filtrar productos que contengan 'gluten-free' o 'gluten free' en el nombre
gf_products = products.filter(
    F.lower(F.col("product_name")).contains("gluten-free") | 
    F.lower(F.col("product_name")).contains("gluten free")
)

gf_products.show(5)

+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
+----------+------------+--------+-------------+



❓ 2️⃣ Use the `orders` and `orderContents` tables to identify the users that have previously ordered a gluten free product. Return a table with all the user id's and a column with a binary value indicating whether an user has bought a gluten free product.

In [0]:
pass  # YOUR CODE HERE
# 1. Obtener lista de usuarios únicos que han comprado productos GF
users_who_bought_gf = orders.join(orderContents, "order_id") \
    .join(gf_products, "product_id") \
    .select("user_id").distinct() \
    .withColumn("bought_gluten_free", F.lit(1))

# 2. Crear tabla con todos los usuarios y un 0 si no han comprado GF
final_gf_user_table = orders.select("user_id").distinct() \
    .join(users_who_bought_gf, "user_id", "left") \
    .fillna(0, subset=["bought_gluten_free"])

final_gf_user_table.show(5)

+-------+------------------+
|user_id|bought_gluten_free|
+-------+------------------+
|  13068|                 0|
|  12422|                 0|
|  22372|                 0|
|  22068|                 0|
|  26081|                 0|
+-------+------------------+
only showing top 5 rows
